In [2]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import polars as pl

sys.path.append('../')

from data_pipeline import load_all_raw_data

from data_analysis import (
    filter_data_until_date, temporal_split_data, 
)

from plots.data_plot import (
    plot_user_analysis, plot_temporal_analysis, 
    plot_station_analysis, plot_activity_heatmap, 
    print_summary_statistics
)

from preprocess import (
    analyze_users_for_visualization, 
    analyze_trips_for_visualization
)

from weather_data import WeatherDataCollector


In [4]:
trips_with_weather = pd.read_csv('../../data/processed/trips_with_weather.csv')

In [5]:
# Convert trips_with_weather to parquet format for more efficient storage and faster loading
#trips_with_weather.to_parquet('../data/processed/trips_with_weather.parquet', compression='snappy')


In [3]:
trips_with_weather = pd.read_parquet('../data/processed/trips_with_weather.parquet')


AttributeError: Can only use .dt accessor with datetimelike values

In [8]:
trips_with_weather['fecha_origen_recorrido'] = pd.to_datetime(trips_with_weather['fecha_origen_recorrido'])
trips_with_weather = trips_with_weather[trips_with_weather['fecha_origen_recorrido'].dt.year >= 2023]

In [12]:
trips_with_weather.isna().sum()

id_recorrido                                 0
duracion_recorrido                           0
fecha_origen_recorrido                       0
id_estacion_origen                           0
nombre_estacion_origen                       0
direccion_estacion_origen                    0
long_estacion_origen                         0
lat_estacion_origen                          0
fecha_destino_recorrido                      0
id_estacion_destino                          2
nombre_estacion_destino                      2
direccion_estacion_destino                   2
long_estacion_destino                        2
lat_estacion_destino                         2
id_usuario                                   0
modelo_bicicleta                             0
genero                                   19625
weather_temperature_2m                       0
weather_relative_humidity_2m                 0
weather_dew_point_2m                         0
weather_apparent_temperature                 0
weather_preci

In [16]:
from data_analysis import engineer_ecobici_features

df_feat = engineer_ecobici_features(trips_with_weather)

df_feat.describe()

🚴‍♂️  Parametrized Feature Engineering Pipeline (11 steps)
  Checkpoints directory: 'feature_checkpoints/'
  Input raw data shape: (4777560, 48)
  -> Memory usage after initialization: 4756.4 MB
[Step 1/11] Converting to Polars and normalizing dates...


  -> Saved Polars conversion checkpoint: step01_polars_conversion.parquet
[Step 2/11] Creating departures table with lags & user profiles...


  -> Saved departures table checkpoint: step02_departures.parquet
[Step 3/11] Creating arrivals table with lags...


  -> Saved arrivals table checkpoint: step03_arrivals.parquet
[Step 4/11] Creating shifted arrivals table for target...


  -> Saved shifted arrivals checkpoint: step04_shifted_arrivals.parquet
[Step 5/11] Creating enhanced weather table...


...

  -> Memory usage after after neighbor demand calculation: 6243.2 MB


Final cleanup:  80%|████████  | 4/5 [00:15<00:04,  4.44s/it]      

  -> Memory usage after after lag features: 6208.0 MB


  -> Memory usage after after final cleanup: 10272.5 MB
  -> Saved final features checkpoint: step11_final.parquet
  -> Memory usage after step 11 complete: 10887.9 MB
  ✅ Saved final result: feature_checkpoints\df_feat_final_result.parquet


,station_id,ts_start,dep_last_DT,trip_dur_mean_last_DT,model_FIT_cnt,model_ICONIC_cnt,share_male,share_female,share_other,dep_lag_1,...,cos_month,is_weekend,payday_flag,vacation_season,peak_commute,dep_ma_24h,dep_std_24h,dep_ratio_DT_24h,near_dep_sum_DT,near_dep_lag_1
count,11118723.0,11118723,11118723.0,2669829.0,11118723.0,11118723.0,2669829.0,2669829.0,2669829.0,11118723.0,...,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0,11118723.0
mean,288.230971,2023-10-31 23:29:59.999999,0.429684,1342.869213,0.291422,0.138262,0.614462,0.297859,0.083488,0.429644,...,-0.120403,0.284515,0.065792,0.203954,0.178871,1.341704,0.684936,0.15173,2.148099,2.148091
min,2.0,2023-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,144.0,2023-06-01 23:30:00,0.0,606.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.866025,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
50%,273.0,2023-10-31 23:30:00,0.0,940.0,0.0,0.0,0.75,0.0,0.0,0.0,...,-0.5,0.0,0.0,0.0,0.0,1.384615,0.617213,0.0,1.0,1.0
75%,453.0,2024-03-31 23:30:00,0.0,1431.0,0.0,0.0,1.0,0.5,0.0,0.0,...,0.5,1.0,0.0,0.0,0.0,1.761905,1.032796,0.0,3.0,3.0
max,556.0,2024-08-30 23:00:00,31.0,3243356.0,27.0,22.0,1.0,1.0,1.0,31.0,...,1.0,1.0,1.0,1.0,1.0,25.0,17.67767,6.666667,64.0,64.0
std,169.68237,NaN,1.000371,10505.095625,0.758124,0.453819,0.423323,0.396641,0.242073,1.00036,...,0.698261,0.451183,0.247918,0.402935,0.383244,0.786069,0.65635,0.327373,3.083676,3.08368


In [9]:
df_feat = pl.read_parquet(r"C:\Users\xxx\Documents\GitHub\EcoBici-AI\notebooks\feature_checkpoints\df_feat_final_result.parquet")

In [4]:
df_feat.columns

['station_id',
 'ts_start',
 'dep_last_DT',
 'trip_dur_mean_last_DT',
 'model_FIT_cnt',
 'model_ICONIC_cnt',
 'share_male',
 'share_female',
 'share_other',
 'dep_lag_1',
 'dep_lag_2',
 'dep_lag_3',
 'dep_lag_4',
 'dep_lag_5',
 'dep_lag_6',
 'arr_last_DT',
 'arr_lag_1',
 'arr_lag_2',
 'arr_lag_3',
 'arr_lag_4',
 'arr_lag_5',
 'arr_lag_6',
 'y_arrivals_next_DT',
 'weather_temperature_2m',
 'weather_relative_humidity_2m',
 'weather_dew_point_2m',
 'weather_apparent_temperature',
 'weather_precipitation',
 'weather_rain',
 'weather_weather_code',
 'weather_pressure_msl',
 'weather_surface_pressure',
 'weather_cloud_cover',
 'weather_cloud_cover_low',
 'weather_cloud_cover_mid',
 'weather_cloud_cover_high',
 'weather_et0_fao_evapotranspiration',
 'weather_vapour_pressure_deficit',
 'weather_wind_speed_10m',
 'weather_wind_speed_100m',
 'weather_wind_direction_10m',
 'weather_wind_direction_100m',
 'weather_wind_gusts_10m',
 'weather_soil_temperature_0_to_7cm',
 'weather_soil_temperature_7_

In [5]:
df_feat.shape

(11118723, 80)

In [10]:
code_cols = ["ts_start"]

feat_cols = ['station_id', 'dep_last_DT', 'trip_dur_mean_last_DT',
       'model_FIT_cnt', 'model_ICONIC_cnt', 'share_male', 'share_female',
       'share_other', 'dep_lag_1', 'dep_lag_2', 'dep_lag_3', 'dep_lag_4',
       'dep_lag_5', 'dep_lag_6', 'arr_last_DT', 'arr_lag_1', 'arr_lag_2',
       'arr_lag_3', 'arr_lag_4', 'arr_lag_5', 'arr_lag_6', 'weather_temperature_2m',
       'weather_relative_humidity_2m', 'weather_dew_point_2m',
       'weather_apparent_temperature', 'weather_precipitation', 'weather_rain',
       'weather_pressure_msl', 'weather_surface_pressure', 'weather_cloud_cover',
       'weather_cloud_cover_low', 'weather_cloud_cover_mid',
       'weather_cloud_cover_high', 'weather_et0_fao_evapotranspiration',
       'weather_vapour_pressure_deficit',
       'weather_soil_temperature_0_to_7cm',
       'weather_soil_moisture_0_to_7cm', 'weather_sunshine_duration',
       'weather_is_day', 'weather_direct_radiation', 'precip_flag',
       'wind_dir_sin', 'wind_dir_cos', 'weather_code_cat_Clear',
       'weather_code_cat_Clouds', 'weather_code_cat_Drizzle',
       'weather_code_cat_Rain', 'weather_code_cat_null',
       'precipitation_lag_1h', 'rain_lag_1h', 'is_holiday_ar', 'sin_hour',
       'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month',
       'is_weekend', 'payday_flag', 'vacation_season', 'peak_commute',
       'dep_ma_24h', 'dep_std_24h', 'dep_ratio_DT_24h', 'near_dep_sum_DT',
       'near_dep_lag_1']

target_col = 'y_arrivals_next_DT'


In [11]:
df_feat = df_feat[code_cols + feat_cols + [target_col]]

QUEDA DROPPEAR ALGUNAS COLUMNAS INNECESARIAS DE WEATHER Y ESTAMOS!

In [15]:
df_feat.sample(5)

ts_start,station_id,dep_last_DT,trip_dur_mean_last_DT,model_FIT_cnt,model_ICONIC_cnt,share_male,share_female,share_other,dep_lag_1,dep_lag_2,dep_lag_3,dep_lag_4,dep_lag_5,dep_lag_6,arr_last_DT,arr_lag_1,arr_lag_2,arr_lag_3,arr_lag_4,arr_lag_5,arr_lag_6,weather_temperature_2m,weather_relative_humidity_2m,weather_dew_point_2m,weather_apparent_temperature,weather_precipitation,weather_rain,weather_pressure_msl,weather_surface_pressure,weather_cloud_cover,weather_cloud_cover_low,weather_cloud_cover_mid,weather_cloud_cover_high,weather_et0_fao_evapotranspiration,weather_vapour_pressure_deficit,weather_soil_temperature_0_to_7cm,weather_soil_moisture_0_to_7cm,weather_sunshine_duration,weather_is_day,weather_direct_radiation,precip_flag,wind_dir_sin,wind_dir_cos,weather_code_cat_Clear,weather_code_cat_Clouds,weather_code_cat_Drizzle,weather_code_cat_Rain,weather_code_cat_null,precipitation_lag_1h,rain_lag_1h,is_holiday_ar,sin_hour,cos_hour,sin_dow,cos_dow,sin_month,cos_month,is_weekend,payday_flag,vacation_season,peak_commute,dep_ma_24h,dep_std_24h,dep_ratio_DT_24h,near_dep_sum_DT,near_dep_lag_1,y_arrivals_next_DT
datetime[μs],i64,u32,f64,u32,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,u8,u8,u8,u8,u8,f64,f64,i8,f32,f32,f32,f32,f32,f32,i8,i8,i8,i8,f32,f32,f32,u32,u32,u32
2023-10-07 17:30:00,508,4,3270.5,4,0,0.5,0.25,0.25,1,1,1,1,1,1,1,1,2,1,1,1,1,21.211,41.923355,7.761,20.250507,0.0,0.0,1019.5,1017.2549,0.0,0.0,0.0,0.0,0.5032366,1.4639902,25.261,0.217,3600.0,1.0,708.0,0,0.527362,0.84964,1,0,0,0,0,0.0,0.0,0,-0.965926,-0.258819,-0.781832,0.62349,-0.866025,0.5,1,0,0,0,1.75,1.5,1.454545,0,3,2
2024-05-20 08:00:00,333,1,733.0,1,0,0.0,0.0,1.0,1,1,1,1,2,2,0,0,0,0,0,0,0,8.711,82.013435,5.811,5.5925484,0.0,0.0,1021.4,1019.0511,2.0,0.0,0.0,2.0,0.012759,0.202894,7.461,0.411,0.0,0.0,0.0,0,-0.278059,0.960564,1,0,0,0,0,0.0,0.0,0,0.866025,-0.5,0.781832,0.62349,0.5,-0.866025,0,0,0,1,1.0,0.0,0.5,0,2,0
2024-02-08 02:30:00,254,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,28.511,78.75502,24.461,33.00501,0.0,0.0,1010.2,1008.0291,91.0,0.0,6.0,90.0,0.039023,0.826377,31.011,0.162,0.0,0.0,0.0,0,0.99846,0.055469,0,1,0,0,0,0.0,0.0,0,0.5,0.866025,-0.433884,-0.900969,0.866025,0.5,0,0,0,0,2.461539,1.570219,0.0,1,1,0
2023-10-08 04:00:00,49,0,null,0,0,null,null,null,0,0,0,0,0,0,0,0,0,0,0,0,0,14.211,58.39774,6.161,11.799513,0.0,0.0,1018.3,1016.003,0.0,0.0,0.0,0.0,0.04098,0.675042,14.561001,0.211,0.0,0.0,0.0,0,-0.68875,0.724998,1,0,0,0,0,0.0,0.0,0,0.866025,0.5,-2.4493e-16,1.0,-0.866025,0.5,0,0,0,0,0.0,0.0,0.0,0,0,0
2023-08-10 19:30:00,60,0,null,0,0,null,null,null,0,0,0,0,0,0,1,1,1,1,1,1,3,17.111,80.408844,13.711,16.763878,0.0,0.0,1007.7,1005.44965,100.0,11.0,31.0,100.0,0.073553,0.382532,17.161001,0.382,0.0,1.0,2.0,0,0.169906,0.98546,0,1,0,0,0,0.0,0.0,0,-0.965926,0.258819,-0.433884,-0.900969,-0.866025,-0.5,0,0,0,1,1.9375,1.236595,0.0,3,6,0


In [14]:
"ts_start" in df_feat.columns

True

In [16]:
from data_analysis import pivot_features_for_global_model

df_feat_pivot = pivot_features_for_global_model(df_feat)

🔄 Pivoting data to wide format for global model...

[Step 1/3] Identifying columns...
  -> Identified 25 station-specific and 40 global features.

[Step 2/3] Constructing feature (X) matrix...
  -> Loading station-specific features from checkpoint: feature_checkpoints\X_station_wide.parquet
  -> Extracting global features...
  -> Joining global and station-specific features...
  -> Saving final features matrix: feature_checkpoints\X_wide.parquet

[Step 3/3] Constructing target (y) matrix...
  -> Pivoting target variable...
  -> Saving checkpoint: feature_checkpoints\y_wide_unaligned.parquet
  -> Aligning target matrix with feature matrix timestamps...
  -> Saving final target matrix: feature_checkpoints\y_wide.parquet
--------------------------------------------------
  Original long shape: (11118723, 68)
  Pivoted X_wide shape: (29183, 9566) -> saved to X_wide.parquet
  Pivoted y_wide shape: (29183, 382) -> saved to y_wide.parquet
✅ Pivoting complete.


In [14]:
X_wide = pl.read_parquet(r"C:\Users\xxx\Documents\GitHub\EcoBici-AI\notebooks\feature_checkpoints\X_wide.parquet")

In [20]:
X_wide.shape

(29183, 9566)

In [19]:
X_wide.columns

['ts_start',
 'weather_temperature_2m',
 'weather_relative_humidity_2m',
 'weather_dew_point_2m',
 'weather_apparent_temperature',
 'weather_precipitation',
 'weather_rain',
 'weather_pressure_msl',
 'weather_surface_pressure',
 'weather_cloud_cover',
 'weather_cloud_cover_low',
 'weather_cloud_cover_mid',
 'weather_cloud_cover_high',
 'weather_et0_fao_evapotranspiration',
 'weather_vapour_pressure_deficit',
 'weather_soil_temperature_0_to_7cm',
 'weather_soil_moisture_0_to_7cm',
 'weather_sunshine_duration',
 'weather_is_day',
 'weather_direct_radiation',
 'precip_flag',
 'wind_dir_sin',
 'wind_dir_cos',
 'weather_code_cat_Clear',
 'weather_code_cat_Clouds',
 'weather_code_cat_Drizzle',
 'weather_code_cat_Rain',
 'weather_code_cat_null',
 'precipitation_lag_1h',
 'rain_lag_1h',
 'is_holiday_ar',
 'sin_hour',
 'cos_hour',
 'sin_dow',
 'cos_dow',
 'sin_month',
 'cos_month',
 'is_weekend',
 'payday_flag',
 'vacation_season',
 'peak_commute',
 'dep_last_DT_2',
 'dep_last_DT_3',
 'dep_last

In [4]:
y_wide = pl.read_parquet(r"C:\Users\xxx\Documents\GitHub\EcoBici-AI\notebooks\feature_checkpoints\y_wide.parquet")

In [5]:
y_wide.shape

(29183, 382)

In [18]:
import polars as pl

def check_nans(df: pl.DataFrame, name: str = "df") -> None:
    # Handle datetime columns separately from numeric columns
    datetime_cols = df.select(pl.col(pl.Datetime)).columns
    numeric_cols = [col for col in df.columns if col not in datetime_cols]
    
    # Get NaN counts for numeric columns
    per_col_numeric = (df.select([pl.col(col).is_nan().sum() for col in numeric_cols])
                      .row(0) if numeric_cols else [])
    
    # Get null counts for datetime columns 
    per_col_datetime = (df.select([pl.col(col).is_null().sum() for col in datetime_cols])
                       .row(0) if datetime_cols else [])
    
    # Combine results
    per_col = list(per_col_numeric) + list(per_col_datetime)
    total_nans = sum(per_col)

    if total_nans == 0:
        print(f"✅ {name}: no NaN/null values found.")
    else:
        print(f"⚠️  {name}: {total_nans:,} NaN/null values detected.")
        # Create DataFrame of columns with NaNs
        nan_cols = pl.DataFrame({
            "column": df.columns,
            "nans": per_col
        })
        (nan_cols
         .filter(pl.col("nans") > 0)
         .sort("nans", descending=True)
         .show())

# ------------------------------------------------------------------
check_nans(X_wide, "X_wide") 
check_nans(y_wide, "y_wide")

✅ X_wide: no NaN/null values found.
✅ y_wide: no NaN/null values found.
